# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# =========================
# 1. Connect to internship repo
# =========================

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            "requirements.txt"
        ],
        check=True
    )
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# =========================
# 2. Hugging Face access
# =========================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

print("HF_TOKEN found.")

# =========================
# 3. DuckDB
# =========================

import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face.")

# =========================
# 4. Load March 2026
# =========================

march_df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        client_has_gsc,
        client_has_ga4,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        scroll_events,
        sessions_ai,
        sessions_paid,
        sessions_direct,
        sessions_social,
        sessions_organic,
        month
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("March 2026 shape:", march_df.shape)
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

display(march_df.head())

HF_TOKEN found.
DuckDB connected to Hugging Face.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 shape: (9841378, 17)
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai,sessions_paid,sessions_direct,sessions_social,sessions_organic,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I use a Random Forest regression model as the capstone model.

The refresh lane needs a ranked decision-support signal rather than an automatic action. Random Forest is suitable for this exploratory modeling task because it can represent non-linear relationships and interactions between search and engagement signals without requiring a linear relationship.

The modeled outcome is observed `scroll_events`, used as an engagement proxy. This is not treated as proof that a content refresh causes more engagement.

The model is compared with the transparent Week-4 baseline on the same held-out data and the same evaluation metric.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a client-grouped split so observations from the same client do not appear in both training and test data.

This reduces the risk that the model learns client-specific patterns that would make the test result look artificially strong.

The test set is held out before model fitting and is used only for final comparison.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

model_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "sessions_organic",
    "scroll_events"
]

model_df = march_df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id"
    ] + model_cols
].copy()

# Keep rows with a known target
model_df = model_df.dropna(
    subset=["scroll_events"]
).copy()

# Numeric conversion
for col in model_cols:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    )

# Fill feature missingness with training-independent median
# for this baseline experiment.
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "sessions_organic"
]

model_df = model_df.dropna(
    subset=["gsc_impressions", "gsc_clicks", "gsc_sum_position"]
).copy()

X = model_df[feature_cols].copy()
y = model_df["scroll_events"].copy()
groups = model_df["client_hash_id"].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

print("Client overlap:",
      len(
          set(groups.iloc[train_idx])
          & set(groups.iloc[test_idx])
      ))

Train rows: 4879304
Test rows: 1943333
Train clients: 34
Test clients: 9
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=150,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_pred = model.predict(X_test)

print("Model trained.")
print("Predictions:", len(model_pred))

Model trained.
Predictions: 1943333


In [4]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from scipy.stats import spearmanr

model_mae = mean_absolute_error(
    y_test,
    model_pred
)

model_rmse = root_mean_squared_error(
    y_test,
    model_pred
)

model_spearman = spearmanr(
    y_test,
    model_pred
).statistic

print("Model MAE:", model_mae)
print("Model RMSE:", model_rmse)
print("Model Spearman:", model_spearman)

Model MAE: 0.08165166096808515
Model RMSE: 0.5400442125262545
Model Spearman: 0.2759058767456245


In [5]:
test_baseline = model_df.iloc[test_idx].copy()

test_baseline["impression_rank"] = (
    test_baseline["gsc_impressions"].rank(pct=True)
)

test_baseline["click_rank"] = (
    test_baseline["gsc_clicks"].rank(pct=True)
)

test_baseline["position_rank"] = (
    1 - test_baseline["gsc_sum_position"].rank(pct=True)
)

test_baseline["baseline_score"] = (
    0.40 * test_baseline["impression_rank"]
    + 0.40 * test_baseline["click_rank"]
    + 0.20 * test_baseline["position_rank"]
)

baseline_score = test_baseline["baseline_score"]

baseline_spearman = spearmanr(
    test_baseline["scroll_events"],
    baseline_score
).statistic

print("Baseline Spearman:", baseline_spearman)

Baseline Spearman: 0.2577030011089898


In [6]:
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "spearman": [
        baseline_spearman,
        model_spearman
    ],
    "mae": [
        np.nan,
        model_mae
    ],
    "rmse": [
        np.nan,
        model_rmse
    ]
})

display(comparison)

,method,spearman,mae,rmse
0,Week-4 baseline,0.257703,NaN,NaN
1,Random Forest,0.275906,0.081652,0.540044


## Model comparison

The baseline and model are evaluated on the same held-out client groups.

Spearman correlation is used for the direct baseline comparison because the Week-4 baseline is a ranking score rather than a calibrated prediction.

The Random Forest also reports MAE and RMSE because it produces a numeric engagement prediction.

The comparison should be interpreted as a benchmark on this observed March 2026 slice, not as evidence that the model will improve future content performance.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

display(importance)

,feature,importance
0,sessions_organic,0.814310
1,gsc_impressions,0.088140
2,gsc_sum_position,0.080506
3,gsc_clicks,0.017044


## Feature interpretation

Feature importance shows which available inputs the Random Forest used most strongly for its observed engagement predictions.

These are model-association measures, not causal effects.

A high importance value does not mean changing that feature will necessarily improve engagement.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
error_analysis = X_test.copy()

error_analysis["actual_scroll_events"] = y_test.values
error_analysis["predicted_scroll_events"] = model_pred

error_analysis["absolute_error"] = (
    error_analysis["actual_scroll_events"]
    - error_analysis["predicted_scroll_events"]
).abs()

error_analysis = error_analysis.sort_values(
    "absolute_error",
    ascending=False
)

display(error_analysis.head(10))

,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_organic,actual_scroll_events,predicted_scroll_events,absolute_error
5044300,2225,2,67267,2,93,1.565271,91.434729
5043736,5437,3,184683,3,87,0.715161,86.284839
5289454,5558,7,182829,8,66,1.638612,64.361388
5290018,2837,1,79926,0,61,0.113070,60.88693
4500131,514,1,7601,1,60,0.123866,59.876134
3616314,551,1,21100,0,59,0.130248,58.869752
5289618,781,3,20727,0,55,0.154459,54.845541
5043812,27,0,1084,0,54,0.049891,53.950109
4811302,2146,2,56401,0,53,0.103567,52.896433
5291540,144,4,1448,0,52,0.117241,51.882759


In [9]:
over_predictions = error_analysis.copy()

over_predictions["signed_error"] = (
    over_predictions["predicted_scroll_events"]
    - over_predictions["actual_scroll_events"]
)

display(
    over_predictions.sort_values(
        "signed_error",
        ascending=False
    ).head(10)
)

,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_organic,actual_scroll_events,predicted_scroll_events,absolute_error,signed_error
1692466,309,4,4739,79,0,25.411752,25.411752,25.411752
90984,1379,2,5036,97,0,25.411752,25.411752,25.411752
1691316,1107,3,4955,77,0,25.262925,25.262925,25.262925
90604,3051,23,15527,91,0,24.565023,24.565023,24.565023
5766558,96,0,899,76,1,25.262925,24.262925,24.262925
4814953,1473,29,7550,130,2,25.411752,23.411752,23.411752
9117423,6878,138,53164,222,1,23.619047,22.619047,22.619047
9158761,6539,69,44478,77,1,23.444391,22.444391,22.444391
91164,3502,47,18181,87,2,24.359865,22.359865,22.359865
8862941,5959,97,46125,138,2,23.619047,21.619047,21.619047


In [10]:
under_predictions = error_analysis.copy()

under_predictions["signed_error"] = (
    under_predictions["predicted_scroll_events"]
    - under_predictions["actual_scroll_events"]
)

display(
    under_predictions.sort_values(
        "signed_error",
        ascending=True
    ).head(10)
)

,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_organic,actual_scroll_events,predicted_scroll_events,absolute_error,signed_error
5044300,2225,2,67267,2,93,1.565271,91.434729,-91.434729
5043736,5437,3,184683,3,87,0.715161,86.284839,-86.284839
5289454,5558,7,182829,8,66,1.638612,64.361388,-64.361388
5290018,2837,1,79926,0,61,0.113070,60.88693,-60.88693
4500131,514,1,7601,1,60,0.123866,59.876134,-59.876134
3616314,551,1,21100,0,59,0.130248,58.869752,-58.869752
5289618,781,3,20727,0,55,0.154459,54.845541,-54.845541
5043812,27,0,1084,0,54,0.049891,53.950109,-53.950109
4811302,2146,2,56401,0,53,0.103567,52.896433,-52.896433
5291540,144,4,1448,0,52,0.117241,51.882759,-51.882759


## 4. Errors and interpretation

The largest errors show where the model's engagement estimates differ most from the observed values.

The model can be wrong when the available search signals do not capture other factors affecting engagement, when source availability differs, or when individual pages behave differently from the broader client-group patterns.

The model should therefore be treated as decision-support rather than an automatic refresh decision.

The feature-importance output is useful for understanding the model's behavior, but it should not be interpreted as causal evidence.

In [11]:
final_summary = pd.DataFrame({
    "metric": [
        "Spearman ranking correlation",
        "MAE",
        "RMSE"
    ],
    "week4_baseline": [
        baseline_spearman,
        np.nan,
        np.nan
    ],
    "random_forest": [
        model_spearman,
        model_mae,
        model_rmse
    ]
})

display(final_summary)

,metric,week4_baseline,random_forest
0,Spearman ranking correlation,0.257703,0.275906
1,MAE,NaN,0.081652
2,RMSE,NaN,0.540044


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.